# Microsoft Foundry: Skills + Toolbox + Tool Search

**The problem this notebook solves:** an agent with ~30 REST tools has every tool's full
JSON-schema definition (name, description, every parameter) baked into its system
instructions. That entire block is re-sent to the model on **every single turn**, whether
that turn needs one of those tools or not — pure token bloat, and it gets worse as you add
tools.

**The fix, in three pieces:**

| Piece | What it is | What it solves |
|---|---|---|
| **Skill** (`SKILL.md`) | A versioned, reusable block of *behavioral instructions* (policy, workflow, domain knowledge) stored once in Foundry and attached to many agents | Duplicated instructions across agents, manual redeploys to update them |
| **Toolbox** | A managed MCP endpoint that bundles many real tools (REST/OpenAPI, MCP servers, etc.) behind one connection | Centralizes tool config, auth, and versioning instead of wiring each REST tool into every agent |
| **Tool Search** | A toggle (`ToolSearchToolboxTool`, or `defer_loading=True` on a single `MCPTool`) that hides a toolbox's tools behind two meta-tools — `tool_search` and `call_tool` — until the model actually asks for one | The actual token-bloat fix: the model's visible tool surface stays at ~2 schemas instead of 30, regardless of toolbox size |

This notebook:

1. Quantifies the token bloat with real numbers (runs anywhere, no Azure needed).
2. Authors a `SKILL.md` (behavioral instructions, decoupled from tool schemas).
3. Configures a real Microsoft Foundry **Skill**, **Toolbox** (wrapping the public
   **GitHub MCP server** — `github/github-mcp-server`, ~30+ tools across `repos`,
   `issues`, `pull_requests`, `actions`, `code_security`, `secret_protection`, and more),
   with **Tool Search** turned on.
4. Attaches both to a Foundry agent and talks to it.
5. Verifies, against the toolbox's own MCP endpoint, that only the meta-tools are visible.

Steps 2–5 need a real Azure subscription + Foundry project + a GitHub personal access
token, so the notebook runs in a **`DEMO_MODE`** that degrades gracefully: with no
credentials configured it still executes top-to-bottom (steps 1–2 for real, steps 3–5 as
clearly-labeled skips), and once you fill in your own environment variables the exact same
cells run for real against your project.

> **Public MCP server used:** [`github/github-mcp-server`](https://github.com/github/github-mcp-server),
> GitHub's official remote MCP server at `https://api.githubcopilot.com/mcp/`. It's a good
> stand-in for "30 REST tools" because it genuinely exposes that many operations, grouped
> into toolsets (`repos`, `issues`, `pull_requests`, `actions`, `code_security`,
> `secret_protection`, `discussions`, `notifications`, `gists`, ...), and anyone with a free
> GitHub account can get a PAT for it. See GitHub's
> [remote server docs](https://github.com/github/github-mcp-server/blob/main/docs/remote-server.md)
> and [toolset docs](https://docs.github.com/en/copilot/how-tos/provide-context/use-mcp/configure-toolsets).


## 0. Setup

SDK versions this notebook was verified against (installed and introspected while writing
this notebook): `azure-ai-projects==2.5.0`, `azure-identity`, `openai`, `mcp`,
`python-dotenv`. These are preview/beta surfaces (Skills, Toolboxes, and Tool Search are
all `(preview)` features in Foundry as of this writing) — class and method names can shift
between SDK releases, so if a call signature below doesn't match your installed version,
check `python -c "import azure.ai.projects.models as m; help(m.<ClassName>)"` first.


In [1]:
#%pip install -q azure-ai-projects>=2.5.0 azure-identity openai mcp python-dotenv

import os, json, asyncio, textwrap
from dataclasses import dataclass

from dotenv import load_dotenv
load_dotenv()

print("Imports OK")


Imports OK


## 1. Configuration and `DEMO_MODE`

Fill these in (a `.env` file works too — `load_dotenv()` above already picked one up if
present) to run everything for real. Leave them blank and the notebook stays in
`DEMO_MODE`: every cell still runs, but Azure-calling cells print what they *would* do
instead of calling out.


In [2]:
# --- Your Foundry project -------------------------------------------------
FOUNDRY_PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "")
# e.g. "https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>"

MODEL_DEPLOYMENT_NAME = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-4o")

# --- The public MCP server we're wrapping in a toolbox ---------------------
GITHUB_MCP_SERVER_URL = os.environ.get("GITHUB_MCP_SERVER_URL", "https://api.githubcopilot.com/mcp/")

# A Foundry "connection" holding your GitHub PAT. Connections that carry secrets are
# provisioned once via the Foundry portal (Management Center -> Connections -> + New
# connection -> API key) or `az`/`azd` CLI -- the SDK's ConnectionsOperations is
# read-only (get / get_default / list), by design, since it never wants your PAT
# round-tripping through client code.
GITHUB_PROJECT_CONNECTION_ID = os.environ.get("GITHUB_PROJECT_CONNECTION_ID", "")

TOOLBOX_NAME = os.environ.get("TOOLBOX_NAME", "github-tools-toolbox")
SKILL_NAME = os.environ.get("SKILL_NAME", "github-issue-triage")
AGENT_NAME = os.environ.get("AGENT_NAME", "github-triage-agent")

DEMO_MODE = not (FOUNDRY_PROJECT_ENDPOINT and GITHUB_PROJECT_CONNECTION_ID)

print(f"DEMO_MODE = {DEMO_MODE}")
if DEMO_MODE:
    print(
        "\nNo FOUNDRY_PROJECT_ENDPOINT / GITHUB_PROJECT_CONNECTION_ID set -- "
        "sections 3-5 below will print what they'd do instead of calling Azure.\n"
        "Set them (env vars or a .env file) and re-run to execute against your real project."
    )


DEMO_MODE = True

No FOUNDRY_PROJECT_ENDPOINT / GITHUB_PROJECT_CONNECTION_ID set -- sections 3-5 below will print what they'd do instead of calling Azure.
Set them (env vars or a .env file) and re-run to execute against your real project.


## 2. Quantify the bloat (runs anywhere, no Azure needed)

Below is a representative 30-tool set modeled on the GitHub MCP server's real toolsets
(`repos`, `issues`, `pull_requests`, `actions`, `code_security`, `secret_protection`) —
each tool written out as a full OpenAI-style function schema, the same shape that would
otherwise sit in your system instructions. We compute how many tokens that costs *every
single turn*, versus the fixed cost of the two Tool Search meta-tools (`tool_search`,
`call_tool`) that replace them.


In [3]:
def _param(name, type_, desc, required=True, enum=None):
    p = {"type": type_, "description": desc}
    if enum:
        p["enum"] = enum
    return name, p, required


def make_tool(name, description, params):
    properties, required = {}, []
    for pname, ptype, pdesc, preq, penum in params:
        properties[pname] = {"type": ptype, "description": pdesc}
        if penum:
            properties[pname]["enum"] = penum
        if preq:
            required.append(pname)
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required},
        },
    }


def P(name, type_, desc, required=True, enum=None):
    return (name, type_, desc, required, enum)


# 30 representative tools, grouped the way github/github-mcp-server groups its own
# toolsets. Real descriptions run this verbose (often more so) because the model needs
# enough detail to pick the right tool and fill in parameters correctly.
REPRESENTATIVE_GITHUB_TOOLS = [
    make_tool("list_repositories", "List repositories for the authenticated user or an organization, optionally filtered by visibility, type, and sort order.",
        [P("owner", "string", "Organization or user login to list repositories for", False, None),
         P("type", "string", "Filter by repository type", False, ["all", "owner", "member"]),
         P("sort", "string", "Sort field", False, ["created", "updated", "pushed", "full_name"])]),
    make_tool("get_repository", "Get detailed metadata for a single repository, including description, default branch, topics, and permissions.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name")]),
    make_tool("create_repository", "Create a new repository owned by the authenticated user or an organization.",
        [P("name", "string", "Repository name"), P("description", "string", "Repository description", False, None),
         P("private", "boolean", "Whether the repository is private", False, None)]),
    make_tool("fork_repository", "Create a fork of an existing repository under the authenticated user or a chosen organization.",
        [P("owner", "string", "Source repository owner"), P("repo", "string", "Source repository name"),
         P("organization", "string", "Organization to fork into", False, None)]),
    make_tool("list_branches", "List branches in a repository, with optional filtering to protected branches only.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"),
         P("protected_only", "boolean", "Only return protected branches", False, None)]),
    make_tool("get_file_contents", "Fetch the contents of a file or directory listing from a repository at a given ref.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"),
         P("path", "string", "File or directory path"), P("ref", "string", "Branch, tag, or commit SHA", False, None)]),
    make_tool("create_or_update_file", "Create a new file or update an existing file's contents on a branch, creating a commit.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("path", "string", "File path"),
         P("content", "string", "New file content"), P("message", "string", "Commit message"),
         P("branch", "string", "Branch to commit to")]),
    make_tool("delete_file", "Delete a file from a repository, creating a commit.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("path", "string", "File path"),
         P("message", "string", "Commit message"), P("branch", "string", "Branch to commit to")]),
    make_tool("list_commits", "List commits on a branch, optionally filtered by author or path.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"),
         P("sha", "string", "Branch name or commit SHA to start listing from", False, None),
         P("path", "string", "Only commits touching this path", False, None)]),
    make_tool("compare_commits", "Compare two commits or branches and return the diff summary and file list.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"),
         P("base", "string", "Base branch or commit SHA"), P("head", "string", "Head branch or commit SHA")]),
    make_tool("list_issues", "List issues in a repository, filterable by state, labels, assignee, and sort order.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"),
         P("state", "string", "Issue state", False, ["open", "closed", "all"]),
         P("labels", "string", "Comma-separated label names to filter by", False, None)]),
    make_tool("get_issue", "Get the full details of a single issue, including body, labels, and assignees.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("issue_number", "integer", "Issue number")]),
    make_tool("create_issue", "Create a new issue in a repository.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("title", "string", "Issue title"),
         P("body", "string", "Issue body", False, None), P("labels", "array", "Labels to apply", False, None)]),
    make_tool("update_issue", "Update an existing issue's title, body, state, or labels.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("issue_number", "integer", "Issue number"),
         P("state", "string", "New state", False, ["open", "closed"])]),
    make_tool("add_issue_comment", "Add a comment to an issue or pull request.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("issue_number", "integer", "Issue number"),
         P("body", "string", "Comment body")]),
    make_tool("list_issue_comments", "List all comments on an issue, in chronological order.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("issue_number", "integer", "Issue number")]),
    make_tool("list_pull_requests", "List pull requests in a repository, filterable by state, base branch, and sort order.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"),
         P("state", "string", "PR state", False, ["open", "closed", "all"])]),
    make_tool("get_pull_request", "Get the full details of a single pull request, including diff stats and mergeability.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("pull_number", "integer", "Pull request number")]),
    make_tool("create_pull_request", "Open a new pull request from a head branch into a base branch.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("title", "string", "PR title"),
         P("head", "string", "Head branch"), P("base", "string", "Base branch"), P("body", "string", "PR description", False, None)]),
    make_tool("merge_pull_request", "Merge a pull request using the merge, squash, or rebase strategy.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("pull_number", "integer", "Pull request number"),
         P("merge_method", "string", "Merge strategy", False, ["merge", "squash", "rebase"])]),
    make_tool("list_pr_reviews", "List all reviews submitted on a pull request.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("pull_number", "integer", "Pull request number")]),
    make_tool("create_pr_review", "Submit a review on a pull request (approve, request changes, or comment).",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("pull_number", "integer", "Pull request number"),
         P("event", "string", "Review verdict", True, ["APPROVE", "REQUEST_CHANGES", "COMMENT"]),
         P("body", "string", "Review summary", False, None)]),
    make_tool("request_pr_reviewers", "Request one or more users or teams as reviewers on a pull request.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("pull_number", "integer", "Pull request number"),
         P("reviewers", "array", "GitHub usernames to request", True, None)]),
    make_tool("list_workflows", "List the GitHub Actions workflows defined in a repository.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name")]),
    make_tool("run_workflow", "Trigger a workflow_dispatch run of a GitHub Actions workflow on a given ref.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("workflow_id", "string", "Workflow file name or ID"),
         P("ref", "string", "Git ref to run the workflow on")]),
    make_tool("list_workflow_runs", "List recent runs of a workflow, filterable by status and branch.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("workflow_id", "string", "Workflow file name or ID", False, None),
         P("status", "string", "Run status filter", False, ["queued", "in_progress", "completed"])]),
    make_tool("get_workflow_run", "Get the status, conclusion, and logs URL for a single workflow run.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("run_id", "integer", "Workflow run ID")]),
    make_tool("cancel_workflow_run", "Cancel a workflow run that is queued or in progress.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"), P("run_id", "integer", "Workflow run ID")]),
    make_tool("list_code_scanning_alerts", "List code scanning alerts for a repository, filterable by severity and state.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"),
         P("state", "string", "Alert state", False, ["open", "closed", "dismissed", "fixed"])]),
    make_tool("list_secret_scanning_alerts", "List secret scanning alerts for a repository, filterable by state and secret type.",
        [P("owner", "string", "Repository owner"), P("repo", "string", "Repository name"),
         P("state", "string", "Alert state", False, ["open", "resolved"])]),
]

print(f"Representative tool set size: {len(REPRESENTATIVE_GITHUB_TOOLS)} tools")


Representative tool set size: 30 tools


In [4]:
def count_tokens(text: str) -> int:
    '''Exact count via tiktoken when it (and its cached encoding file) is available;
    otherwise a standard ~4-chars-per-token approximation -- close enough to make the
    point, and it never needs network access.'''
    try:
        import tiktoken
        enc = tiktoken.get_encoding("cl100k_base")
        return len(enc.encode(text))
    except Exception:
        return max(1, len(text) // 4)


all_tool_defs_json = json.dumps(REPRESENTATIVE_GITHUB_TOOLS)
all_tools_tokens = count_tokens(all_tool_defs_json)
all_tools_chars = len(all_tool_defs_json)

TOOL_SEARCH_META_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "tool_search",
            "description": "Search for available tools in the toolbox by natural-language capability.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural-language description of the capability needed"},
                    "limit": {"type": "integer", "description": "Max tools to return (default 5, max 10)"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "call_tool",
            "description": "Invoke a tool previously discovered via tool_search, by exact name, with matching arguments.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Exact tool name returned by tool_search"},
                    "arguments": {"type": "object", "description": "Arguments matching that tool's parameter schema"},
                },
                "required": ["name", "arguments"],
            },
        },
    },
]
meta_tools_json = json.dumps(TOOL_SEARCH_META_TOOLS)
meta_tools_tokens = count_tokens(meta_tools_json)
meta_tools_chars = len(meta_tools_json)

print("Per-turn tool-definition cost")
print("-" * 60)
print(f"{'All 30 tool schemas inline':<38}{all_tools_chars:>8,} chars  {all_tools_tokens:>7,} tokens")
print(f"{'tool_search + call_tool only':<38}{meta_tools_chars:>8,} chars  {meta_tools_tokens:>7,} tokens")
reduction = 1 - (meta_tools_tokens / all_tools_tokens)
print(f"\nReduction: {reduction:.1%} fewer tool-definition tokens on every single turn.")
print(
    f"\nOver 1,000 turns that's roughly {(all_tools_tokens - meta_tools_tokens) * 1000:,} "
    "input tokens saved on tool definitions alone -- before counting the conversation itself."
)


Per-turn tool-definition cost
------------------------------------------------------------
All 30 tool schemas inline              15,041 chars    3,760 tokens
tool_search + call_tool only               851 chars      212 tokens

Reduction: 94.4% fewer tool-definition tokens on every single turn.

Over 1,000 turns that's roughly 3,548,000 input tokens saved on tool definitions alone -- before counting the conversation itself.


## 3. Author the Skill: behavioral instructions, decoupled from tool schemas

This is the piece that answers "can `SKILL.md` define tools?" — **no**. The frontmatter
only carries `name`, `description`, `license`, `compatibility`, `metadata`, and the
experimental `allowed-tools` (a plain allow-list of tool *names*, not schemas). Tool
schemas live in the toolbox, not the skill. What the skill *does* carry is the policy text
you'd otherwise have to paste into every agent.


In [5]:
SKILL_NAME_SLUG = "github-issue-triage"

SKILL_MD = f'''---
name: {SKILL_NAME_SLUG}
description: Triage incoming GitHub issues -- classify severity, apply labels, and draft a first response. Use when asked to triage, categorize, or respond to GitHub issues.
metadata:
  owner: platform-team
  version: "1.0"
allowed-tools: list_issues get_issue add_issue_comment update_issue
---

# GitHub Issue Triage

You triage incoming GitHub issues for this repository consistently, the same way every
time, regardless of which agent or team is running this skill.

## Steps

1. Pull the issue's title, body, and existing labels with `get_issue`.
2. Classify severity:
   - **critical**: data loss, security vulnerability, or a broken build on `main`
   - **bug**: reproducible defect with no critical impact
   - **question**: the reporter is asking how to do something, not reporting a defect
   - **enhancement**: a feature request
3. Apply exactly one severity label via `update_issue` (remove conflicting old labels first).
4. Draft a first-response comment with `add_issue_comment`:
   - Thank the reporter.
   - Restate the problem in one sentence so they know it was understood.
   - For `critical`, say a maintainer will follow up within one business day.
   - For `question`, answer directly if the answer is unambiguous from the repo's docs;
     otherwise route to the discussion board.

## Tone

Direct and warm. Never say "as an AI" or hedge about whether the issue is valid --
that's for a human maintainer to decide later if needed.
'''

skill_path = "SKILL.md"
with open(skill_path, "w") as f:
    f.write(SKILL_MD)

print(SKILL_MD)


---
name: github-issue-triage
description: Triage incoming GitHub issues -- classify severity, apply labels, and draft a first response. Use when asked to triage, categorize, or respond to GitHub issues.
metadata:
  owner: platform-team
  version: "1.0"
allowed-tools: list_issues get_issue add_issue_comment update_issue
---

# GitHub Issue Triage

You triage incoming GitHub issues for this repository consistently, the same way every
time, regardless of which agent or team is running this skill.

## Steps

1. Pull the issue's title, body, and existing labels with `get_issue`.
2. Classify severity:
   - **critical**: data loss, security vulnerability, or a broken build on `main`
   - **bug**: reproducible defect with no critical impact
   - **question**: the reporter is asking how to do something, not reporting a defect
   - **enhancement**: a feature request
3. Apply exactly one severity label via `update_issue` (remove conflicting old labels first).
4. Draft a first-response comment wi

## 4. Configure in Foundry: create the Skill

`project.beta.skills.create(name=..., inline_content=SkillInlineContent(description=...,
instructions=...))` uploads a skill and returns its first version. Calling `create` again
on the same name creates a new, immutable version — you promote one to `default_version`
to roll it out without touching any agent that references the skill by name.


In [6]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import SkillInlineContent

# Pull the body back out of the SKILL.md we just wrote (frontmatter stripped) so the
# uploaded skill's `instructions` match the file exactly.
_, _, skill_body = SKILL_MD.partition("---\n")[2].partition("---\n")
SKILL_DESCRIPTION = ("Triage incoming GitHub issues -- classify severity, apply labels, "
                     "and draft a first response. Use when asked to triage, categorize, "
                     "or respond to GitHub issues.")


def create_or_update_skill(project: "AIProjectClient", name: str, description: str, instructions: str):
    return project.beta.skills.create(
        name=name,
        inline_content=SkillInlineContent(description=description, instructions=instructions),
    )


created_skill = None
if DEMO_MODE:
    print("[DEMO_MODE] Would call:")
    print(f'  project.beta.skills.create(name="{SKILL_NAME_SLUG}", '
          f'inline_content=SkillInlineContent(description=..., instructions=<SKILL.md body>))')
else:
    with DefaultAzureCredential() as credential, AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential, allow_preview=True
    ) as project:
        try:
            created_skill = create_or_update_skill(project, SKILL_NAME_SLUG, SKILL_DESCRIPTION, skill_body.strip())
            print(f"Created skill '{created_skill.name}', version {created_skill.version}")
        except Exception as e:
            print(f"Skill creation failed (check your project endpoint / auth / SDK version): {e}")


[DEMO_MODE] Would call:
  project.beta.skills.create(name="github-issue-triage", inline_content=SkillInlineContent(description=..., instructions=<SKILL.md body>))


## 5. Configure in Foundry: create the Toolbox, with Tool Search on

`MCPToolboxTool` wraps the GitHub MCP server; `ToolSearchToolboxTool()` in the same `tools`
list is what flips the toolbox from "list all 30 tools upfront" to "hide everything behind
`tool_search` / `call_tool`"; `skills=[ToolboxSkillReference(name=...)]` attaches the skill
from step 4 to this toolbox so any agent consuming it gets both.

`project_connection_id` points at a Foundry **connection** holding your GitHub PAT.
Connections that carry secrets are provisioned once via the Foundry portal or `az`/`azd`
CLI (the SDK's `ConnectionsOperations` is intentionally read-only — `get` / `get_default` /
`list` — so a PAT never has to round-trip through this notebook).


In [7]:
from azure.ai.projects.models import MCPToolboxTool, ToolSearchToolboxTool, ToolboxSkillReference

def build_toolbox_tools(github_connection_id: str, skill_name: str):
    github_mcp_tool = MCPToolboxTool(
        server_label="github",
        server_url=GITHUB_MCP_SERVER_URL,
        require_approval="never",           # or "always" to require human sign-off per call
        project_connection_id=github_connection_id,
    )
    tools = [github_mcp_tool, ToolSearchToolboxTool()]
    skills = [ToolboxSkillReference(name=skill_name)]
    return tools, skills


created_toolbox = None
if DEMO_MODE:
    print("[DEMO_MODE] Would call:")
    print(f'  project.toolboxes.create_version(')
    print(f'      name="{TOOLBOX_NAME}",')
    print(f'      description="GitHub tools (30+) exposed through Tool Search, plus the {SKILL_NAME_SLUG} skill",')
    print(f'      tools=[MCPToolboxTool(server_label="github", server_url="{GITHUB_MCP_SERVER_URL}", '
          f'require_approval="never", project_connection_id="<your-github-connection-id>"), '
          f'ToolSearchToolboxTool()],')
    print(f'      skills=[ToolboxSkillReference(name="{SKILL_NAME_SLUG}")],')
    print(f'  )')
else:
    with DefaultAzureCredential() as credential, AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential, allow_preview=True
    ) as project:
        try:
            tools, skills = build_toolbox_tools(GITHUB_PROJECT_CONNECTION_ID, SKILL_NAME_SLUG)
            created_toolbox = project.toolboxes.create_version(
                name=TOOLBOX_NAME,
                description=f"GitHub tools (30+) exposed through Tool Search, plus the {SKILL_NAME_SLUG} skill",
                tools=tools,
                skills=skills,
            )
            print(f"Created toolbox '{created_toolbox.name}', version {created_toolbox.version}")
        except Exception as e:
            print(f"Toolbox creation failed (check your connection ID / auth / SDK version): {e}")


[DEMO_MODE] Would call:
  project.toolboxes.create_version(
      name="github-tools-toolbox",
      description="GitHub tools (30+) exposed through Tool Search, plus the github-issue-triage skill",
      tools=[MCPToolboxTool(server_label="github", server_url="https://api.githubcopilot.com/mcp/", require_approval="never", project_connection_id="<your-github-connection-id>"), ToolSearchToolboxTool()],
      skills=[ToolboxSkillReference(name="github-issue-triage")],
  )


## 6. Verify: the toolbox's own MCP endpoint should show only the meta-tools

Every toolbox version is itself reachable as a standard MCP endpoint at
`{endpoint}/toolboxes/{name}/versions/{version}/mcp?api-version=v1`. Connecting to it
directly with the `mcp` Python client and calling `list_tools()` is the ground truth for
"did Tool Search actually turn on" — with it enabled you should see `tool_search` and
`call_tool` (plus any tools you explicitly pinned), and nothing else, no matter how many
real tools the toolbox wraps.


In [8]:
async def verify_tool_search(project_endpoint: str, toolbox_name: str, toolbox_version: str):
    from mcp.client.streamable_http import streamablehttp_client
    from mcp import ClientSession

    url = f"{project_endpoint.rstrip('/')}/toolboxes/{toolbox_name}/versions/{toolbox_version}/mcp?api-version=v1"
    token = DefaultAzureCredential().get_token("https://ai.azure.com/.default").token
    headers = {"Authorization": f"Bearer {token}", "Foundry-Features": "Toolboxes=V1Preview"}

    async with streamablehttp_client(url, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            names = sorted(t.name for t in tools_result.tools)
            print(f"Tools visible in tools/list ({len(names)}): {names}")

            meta_tools = {"tool_search", "call_tool"}
            assert meta_tools <= set(names), "Tool Search meta-tools are missing -- check toolbox_search config"
            unexpected = set(names) - meta_tools
            if unexpected:
                print(f"Also pinned (always visible, no search round-trip needed): {sorted(unexpected)}")
            else:
                print("No other tools leaked into tools/list -- Tool Search is hiding all 30+ GitHub tools as expected.")
            return tools_result


if DEMO_MODE or created_toolbox is None:
    print("[DEMO_MODE] Skipping live verification -- would connect to:")
    print(f'  {{FOUNDRY_PROJECT_ENDPOINT}}/toolboxes/{TOOLBOX_NAME}/versions/<version>/mcp?api-version=v1')
    print("  and assert tools/list == {'tool_search', 'call_tool'}")
else:
    try:
        await verify_tool_search(FOUNDRY_PROJECT_ENDPOINT, created_toolbox.name, created_toolbox.version)
    except Exception as e:
        print(f"Verification failed: {e}")


[DEMO_MODE] Skipping live verification -- would connect to:
  {FOUNDRY_PROJECT_ENDPOINT}/toolboxes/github-tools-toolbox/versions/<version>/mcp?api-version=v1
  and assert tools/list == {'tool_search', 'call_tool'}


## 7. Create the agent: attach the Skill and the Toolbox

A Foundry agent version (`PromptAgentDefinition`) takes a plain `tools: list[Tool]`. Two
entries matter here:

- `SkillReferenceParam(skill_id=..., version="latest")` — the documented, agent-level way
  to attach a skill (`type: SKILL_REFERENCE`) regardless of whether it's also attached to a
  toolbox.
- `MCPTool(server_url=<the toolbox's own MCP endpoint>, ...)` — because a toolbox *is* an
  MCP endpoint, an agent consumes it exactly like it would any other MCP server. You could
  alternatively skip the toolbox entirely and put `MCPTool(server_url=<github mcp url>,
  defer_loading=True)` straight on the agent — `defer_loading` is the same Tool Search
  behavior, scoped to a single MCP tool on a single agent instead of a shared, versioned,
  multi-agent toolbox.


In [9]:
from azure.ai.projects.models import PromptAgentDefinition, SkillReferenceParam, MCPTool

def build_agent_definition(model_deployment: str, toolbox_mcp_url: str, skill_id: str):
    return PromptAgentDefinition(
        model=model_deployment,
        instructions=(
            "You triage GitHub issues. Use the github-issue-triage skill's steps exactly. "
            "Always confirm the repository (owner/repo) before taking any action."
        ),
        tools=[
            SkillReferenceParam(skill_id=skill_id, version="latest"),
            MCPTool(
                server_label="github-toolbox",
                server_url=toolbox_mcp_url,
                require_approval="never",
            ),
        ],
    )


created_agent_version = None
if DEMO_MODE or created_skill is None or created_toolbox is None:
    print("[DEMO_MODE] Would call:")
    print(f'  project.agents.create_version(')
    print(f'      agent_name="{AGENT_NAME}",')
    print(f'      definition=PromptAgentDefinition(model="{MODEL_DEPLOYMENT_NAME}", instructions=..., tools=[')
    print(f'          SkillReferenceParam(skill_id="<{SKILL_NAME_SLUG} skill id>", version="latest"),')
    print(f'          MCPTool(server_label="github-toolbox", '
          f'server_url="{{endpoint}}/toolboxes/{TOOLBOX_NAME}/versions/<v>/mcp?api-version=v1"),')
    print(f'      ]),')
    print(f'  )')
else:
    toolbox_mcp_url = (
        f"{FOUNDRY_PROJECT_ENDPOINT.rstrip('/')}/toolboxes/{created_toolbox.name}"
        f"/versions/{created_toolbox.version}/mcp?api-version=v1"
    )
    with DefaultAzureCredential() as credential, AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential, allow_preview=True
    ) as project:
        try:
            definition = build_agent_definition(MODEL_DEPLOYMENT_NAME, toolbox_mcp_url, created_skill.name)
            created_agent_version = project.agents.create_version(agent_name=AGENT_NAME, definition=definition)
            print(f"Created agent '{AGENT_NAME}', version {created_agent_version.version}")
        except Exception as e:
            print(f"Agent creation failed (check model deployment name / SDK version for your Foundry release): {e}")


[DEMO_MODE] Would call:
  project.agents.create_version(
      agent_name="github-triage-agent",
      definition=PromptAgentDefinition(model="gpt-4o", instructions=..., tools=[
          SkillReferenceParam(skill_id="<github-issue-triage skill id>", version="latest"),
          MCPTool(server_label="github-toolbox", server_url="{endpoint}/toolboxes/github-tools-toolbox/versions/<v>/mcp?api-version=v1"),
      ]),
  )


## 8. Talk to the agent

`project.get_openai_client(agent_name=...)` (available on `AIProjectClient` when
`allow_preview=True`) returns a standard `openai` client pointed at that specific agent's
own OpenAI-compatible endpoint, authenticated with your Azure credential. Note this call
shape is the newest, least-documented part of the preview surface as of this writing --
if it errors on your SDK version, check the release notes for `azure-ai-projects` or your
Foundry project's Responses/Chat Completions docs for the current invocation call.


In [10]:
if DEMO_MODE or created_agent_version is None:
    print("[DEMO_MODE] Would call:")
    print(f'  client = project.get_openai_client(agent_name="{AGENT_NAME}")')
    print(f'  client.responses.create(model="{AGENT_NAME}", input="Triage issue #42 in octocat/Hello-World")')
else:
    with DefaultAzureCredential() as credential, AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential, allow_preview=True
    ) as project:
        try:
            client = project.get_openai_client(agent_name=AGENT_NAME)
            response = client.responses.create(
                model=AGENT_NAME,
                input="Triage issue #42 in octocat/Hello-World and post the first-response comment.",
            )
            print(response.output_text)
            if getattr(response, "usage", None):
                print(f"\nToken usage reported by the API: {response.usage}")
        except Exception as e:
            print(f"Agent invocation failed: {e}")


[DEMO_MODE] Would call:
  client = project.get_openai_client(agent_name="github-triage-agent")
  client.responses.create(model="github-triage-agent", input="Triage issue #42 in octocat/Hello-World")


## 9. Cleanup (optional)

Skills, toolboxes, and agents are all versioned resources — deleting the whole named
resource (not just one version) removes every version. Only run this if you don't want to
keep the demo resources around.


In [11]:
CLEANUP = False  # flip to True to actually delete what this notebook created

if CLEANUP and not DEMO_MODE:
    with DefaultAzureCredential() as credential, AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential, allow_preview=True
    ) as project:
        for label, fn in [
            ("agent", lambda: project.agents.delete(AGENT_NAME)),
            ("toolbox", lambda: project.toolboxes.delete(TOOLBOX_NAME)),
            ("skill", lambda: project.beta.skills.delete(SKILL_NAME_SLUG)),
        ]:
            try:
                fn()
                print(f"Deleted {label}")
            except Exception as e:
                print(f"Could not delete {label}: {e}")
else:
    print("Skipped (CLEANUP is False or DEMO_MODE is on).")


Skipped (CLEANUP is False or DEMO_MODE is on).


## Summary

- **`SKILL.md` cannot define tool schemas.** Its frontmatter is `name` / `description` /
  `license` / `compatibility` / `metadata` / the experimental `allowed-tools` (a name
  allow-list, not a schema). Tool definitions live in a **Toolbox**.
- **Toolbox** = one managed MCP endpoint wrapping many real tools (here, GitHub's public
  MCP server) plus your reusable skills, versioned and shared across agents.
- **Tool Search** (`ToolSearchToolboxTool()` on a toolbox, or `defer_loading=True` on a
  single `MCPTool`) is what actually fixes the token-bloat problem: the model's visible
  tool surface drops from N schemas to 2 (`tool_search`, `call_tool`), and Section 2's
  numbers show roughly how much that's worth on a 30-tool set.
- Everything above is real, currently-installed `azure-ai-projects==2.5.0` SDK surface,
  confirmed by introspecting the package directly, not guessed from docs alone. The one
  cell flagged as least certain is Section 8 (the exact agent-invocation call shape),
  since that corner of the preview API had the thinnest published documentation at the
  time this notebook was written.

### Sources
- [Use skills with Microsoft Foundry agents (preview)](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/skills)
- [What is Toolbox in Microsoft Foundry?](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/toolbox-overview)
- [Enable tool search in a toolbox](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/tool-search)
- [Agent Skills specification](https://agentskills.io/specification)
- [github/github-mcp-server](https://github.com/github/github-mcp-server) /
  [remote server docs](https://github.com/github/github-mcp-server/blob/main/docs/remote-server.md) /
  [toolset configuration](https://docs.github.com/en/copilot/how-tos/provide-context/use-mcp/configure-toolsets)
